# SeatGuard — Station 3: Plastic Parts & Component Presence Detection

Trains YOLOv8 on the **PCB Defect Dataset** from Kaggle.  
PCB component presence/absence logic maps directly to Station 3's part detection.

**Runtime:** GPU → Runtime → Change runtime type → T4 GPU

**Original PCB classes → remapped to SeatGuard Station 3 classes:**
| PCB class | Station 3 class | Reason |
|-----------|----------------|--------|
| `missing_hole` | `part_missing` | Component absent |
| `open_circuit` | `part_wrong` | Wrong/broken connection |
| `short` | `part_wrong` | Wrong assembly |
| `spur` | `part_wrong` | Defective part |
| `spurious_copper` | `part_wrong` | Defective part |
| `mouse_bite` | `part_missing` | Missing material |

**Expected output:** `plastic_crack.onnx` (download at end)

## Step 1 — Install dependencies

In [ ]:
!pip install ultralytics kaggle --quiet

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2 — Upload Kaggle API token

Go to [kaggle.com](https://kaggle.com) → Your Profile → Settings → API → **Create New Token**  
It downloads `kaggle.json` to your PC. Upload it when prompted below.

In [ ]:
from google.colab import files
import os

print("Upload your kaggle.json file:")
uploaded = files.upload()   # select kaggle.json from your PC

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle auth configured.")

## Step 3 — Download PCB Defect Dataset

In [ ]:
!kaggle datasets download -d akhatova/pcb-defects -p /content/pcb_raw --unzip

# Check what was downloaded
import os
for root, dirs, files_list in os.walk("/content/pcb_raw"):
    level = root.replace("/content/pcb_raw", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = " " * 2 * (level + 1)
        for f in files_list[:5]:
            print(f"{subindent}{f}")

## Step 4 — Convert to YOLOv8 format with class remapping

The PCB dataset comes with XML annotations (Pascal VOC format).  
This cell converts them to YOLO `.txt` labels and remaps the classes.

In [ ]:
import os
import xml.etree.ElementTree as ET
import shutil
import random
from pathlib import Path

# ── Class remapping: PCB defect → SeatGuard Station 3 ────────────────────────
CLASS_MAP = {
    "missing_hole":     "part_missing",
    "mouse_bite":       "part_missing",
    "open_circuit":     "part_wrong",
    "short":            "part_wrong",
    "spur":             "part_wrong",
    "spurious_copper":  "part_wrong",
}
# Station 3 class index
# part_correct = 0 (will be added as clean images with no labels)
# part_missing = 1
# part_wrong   = 2
CLASS_INDEX = {"part_correct": 0, "part_missing": 1, "part_wrong": 2}
# ─────────────────────────────────────────────────────────────────────────────

DATASET_ROOT = Path("/content/pcb_dataset")
RAW_ROOT = Path("/content/pcb_raw")

# Find images and annotations
# PCB dataset structure can vary — find all jpg/xml pairs
img_paths = sorted(list(RAW_ROOT.rglob("*.jpg")) + list(RAW_ROOT.rglob("*.JPG")) + list(RAW_ROOT.rglob("*.png")))
xml_paths = sorted(RAW_ROOT.rglob("*.xml"))
print(f"Found {len(img_paths)} images and {len(xml_paths)} XML annotation files")

def parse_voc_xml(xml_path):
    """Parse Pascal VOC XML and return (width, height, list of (class_name, xmin, ymin, xmax, ymax))."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    w = int(size.find("width").text)
    h = int(size.find("height").text)
    boxes = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip().lower().replace(" ", "_")
        bndbox = obj.find("bndbox")
        xmin = int(float(bndbox.find("xmin").text))
        ymin = int(float(bndbox.find("ymin").text))
        xmax = int(float(bndbox.find("xmax").text))
        ymax = int(float(bndbox.find("ymax").text))
        boxes.append((name, xmin, ymin, xmax, ymax))
    return w, h, boxes

def voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
    """Convert VOC bbox to YOLO normalised cx,cy,w,h."""
    cx = ((xmin + xmax) / 2) / img_w
    cy = ((ymin + ymax) / 2) / img_h
    bw = (xmax - xmin) / img_w
    bh = (ymax - ymin) / img_h
    return cx, cy, bw, bh

# Build image → xml mapping
xml_by_stem = {p.stem: p for p in xml_paths}

# Split 80/10/10
random.seed(42)
all_imgs = [p for p in img_paths if p.stem in xml_by_stem]
random.shuffle(all_imgs)
n = len(all_imgs)
train_imgs = all_imgs[:int(n * 0.8)]
valid_imgs = all_imgs[int(n * 0.8):int(n * 0.9)]
test_imgs  = all_imgs[int(n * 0.9):]
print(f"Train: {len(train_imgs)}  Valid: {len(valid_imgs)}  Test: {len(test_imgs)}")

skipped = 0
converted = 0

for split, split_imgs in [("train", train_imgs), ("valid", valid_imgs), ("test", test_imgs)]:
    img_out = DATASET_ROOT / split / "images"
    lbl_out = DATASET_ROOT / split / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path in split_imgs:
        xml_path = xml_by_stem.get(img_path.stem)
        if xml_path is None:
            skipped += 1
            continue

        try:
            img_w, img_h, boxes = parse_voc_xml(xml_path)
        except Exception as e:
            skipped += 1
            continue

        yolo_lines = []
        for (raw_cls, xmin, ymin, xmax, ymax) in boxes:
            mapped_cls = CLASS_MAP.get(raw_cls)
            if mapped_cls is None:
                continue   # unknown class, skip box
            cls_id = CLASS_INDEX[mapped_cls]
            cx, cy, bw, bh = voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h)
            yolo_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        # Copy image
        shutil.copy(img_path, img_out / img_path.name)
        # Write label file (empty = no defects = part_correct implicitly)
        lbl_file = lbl_out / (img_path.stem + ".txt")
        lbl_file.write_text("\n".join(yolo_lines))
        converted += 1

print(f"Converted: {converted}  Skipped: {skipped}")

# Write data.yaml
data_yaml_path = str(DATASET_ROOT / "data.yaml")
import yaml
data_cfg = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc": 3,
    "names": ["part_correct", "part_missing", "part_wrong"],
}
with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("\ndata.yaml written:")
print(yaml.dump(data_cfg))

## Step 5 — Train YOLOv8

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data=data_yaml_path,
    epochs=75,
    imgsz=640,
    batch=12,           # reduce to 8 if OOM
    lr0=0.01,
    patience=15,
    augment=True,
    degrees=180.0,      # parts can be any rotation
    fliplr=0.5,
    flipud=0.5,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    perspective=0.001,
    mixup=0.1,
    copy_paste=0.1,
    project="seatguard_station3",
    name="plastic_crack",
    exist_ok=True,
    device=0,
    conf=0.40,
    verbose=True,
)

print("\nTraining complete.")

## Step 6 — Evaluate

In [ ]:
import glob
best_pt = glob.glob("seatguard_station3/plastic_crack*/weights/best.pt")[0]
print(f"Best weights: {best_pt}")

trained_model = YOLO(best_pt)
metrics = trained_model.val(data=data_yaml_path, conf=0.40)

print(f"\nmAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

print("\nPer-class AP50:")
for cls_name, ap in zip(trained_model.names.values(), metrics.box.ap50):
    print(f"  {cls_name:20s}: {ap:.4f}")

## Step 7 — Export to ONNX

In [ ]:
trained_model.export(
    format="onnx",
    imgsz=640,
    opset=17,
    dynamic=True,
    simplify=True,
)

onnx_path = best_pt.replace(".pt", ".onnx")
print(f"ONNX model: {onnx_path}")

import onnx
m = onnx.load(onnx_path)
onnx.checker.check_model(m)
print("ONNX check passed.")

for inp in m.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"Input  '{inp.name}': {shape}")
for out in m.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"Output '{out.name}': {shape}")

## Step 8 — Smoke test

In [ ]:
import glob, random, os
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

val_images = glob.glob("/content/pcb_dataset/valid/images/*")
sample_img = random.choice(val_images)
print(f"Testing on: {sample_img}")

preds = trained_model.predict(sample_img, conf=0.40, verbose=False)
result = preds[0]

img = Image.open(sample_img)
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(img)

colors = {"part_correct": "green", "part_missing": "red", "part_wrong": "orange"}
for box in result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    cls_id = int(box.cls[0])
    cls_name = trained_model.names[cls_id]
    conf = float(box.conf[0])
    color = colors.get(cls_name, "blue")
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor=color, facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1-5, f"{cls_name} {conf:.2f}",
            color=color, fontsize=9, weight="bold")

ax.set_title(f"Detections: {len(result.boxes)}")
ax.axis("off")
plt.tight_layout()
plt.show()

## Step 9 — Download ONNX file

In [ ]:
import shutil
from google.colab import files

output_name = "plastic_crack.onnx"
shutil.copy(onnx_path, output_name)

print(f"File size: {os.path.getsize(output_name) / 1e6:.1f} MB")
files.download(output_name)
print("\nPlace this file at:  models/station_3/plastic_crack.onnx")

## After downloading

1. Copy `plastic_crack.onnx` to `models/station_3/plastic_crack.onnx`
2. Update `configs/stations/station_3_plastic.yaml`:

```yaml
model:
  backend: "onnx"
  model_type: "yolo_presence"
  weights_path: "models/station_3/plastic_crack.onnx"
  version: "v1.0.0"
  threshold: 0.40
  class_names:
    - "part_correct"
    - "part_missing"
    - "part_wrong"
```

3. Bring the model back to the chat and it will be integrated automatically.

## Expected results
| Metric | Acceptable | Good |
|--------|-----------|------|
| mAP50  | > 0.55    | > 0.70 |
| Precision | > 0.60 | > 0.75 |

The PCB dataset has ~693 images — smaller than Station 2's dataset.  
mAP will be lower but still enough for a working baseline.